# SYNERGY+ getting started

This notebook exercises the Python API described in the README, using **only the SYNERGY+ dataset**. It covers:

- Confirming the active dataset variant
- Iterating over all datasets
- Iterating over the train/test split (SYNERGY+ only)
- Working with a single dataset: metadata, labels, citation, summary statistics
- Exporting a dataset to a DataFrame / dict (default, extended, and specific fields)
- Iterating over individual works, with and without Pydantic validation
- Building one combined DataFrame across datasets

In [1]:
import os

# SYNERGY+ is already the default, but we pin it explicitly so this
# notebook always runs against SYNERGY+ regardless of the environment.
os.environ["SYNERGY_SET"] = "synergy+"

from pprint import pprint

import pandas as pd

from synergy_dataset import TEST_SPLIT
from synergy_dataset import WORK_EXTRACTORS
from synergy_dataset import Dataset
from synergy_dataset import iter_datasets


## Dataset variant

Confirm we are indeed running against SYNERGY+.

In [2]:
from synergy_dataset.base import SYNERGY_SET
from synergy_dataset.base import SYNERGY_VERSION

print("SYNERGY_SET:", SYNERGY_SET)
print("SYNERGY_VERSION:", SYNERGY_VERSION)
assert SYNERGY_SET == "synergy+"

SYNERGY_SET: synergy+
SYNERGY_VERSION: 1.0


## Iterating over datasets

`iter_datasets()` yields a `Dataset` object per systematic review in SYNERGY+.

In [3]:
dataset_names = [d.name for d in iter_datasets()]

print(f"{len(dataset_names)} datasets in SYNERGY+")
dataset_names[:10]

114 datasets in SYNERGY+


['Abgaz_2023',
 'Adamo_2021',
 'Ali_2024',
 'Anmarkrud_2021',
 'Aouad_2024',
 'Appenzeller-Herzog_2019',
 'Attai_2022',
 'Bakker-Jacobs_2022',
 'Bech_2019',
 'Bindoli_2024a']

## Iterating over the train/test split

SYNERGY+ ships an official train/test split (`TEST_SPLIT` holds the test set names). `iter_datasets(split=...)` filters accordingly, and raises a `ValueError` for anything other than SYNERGY+.

In [4]:
train_names = [d.name for d in iter_datasets(split="train")]
test_names = [d.name for d in iter_datasets(split="test")]

print(f"train: {len(train_names)} datasets")
print(f"test:  {len(test_names)} datasets")

assert len(train_names) + len(test_names) == len(dataset_names)
assert set(train_names).isdisjoint(test_names)
assert set(test_names) == set(TEST_SPLIT)

test_names[:10]

train: 91 datasets
test:  23 datasets


['Abgaz_2023',
 'Attai_2022',
 'Boersma-van_Dam_2024',
 'Bos_2018',
 'Bosch_2021',
 'Chakraborty_2023',
 'Chueca_2023',
 'Donners_2021',
 'Eggmann_2023',
 'Hall_2011']

## Working with a single dataset

`Dataset(name)` gives access to one review's metadata, labels, and works.

In [5]:
d = Dataset("Appenzeller-Herzog_2019")
d.name, d.name in TEST_SPLIT

('Appenzeller-Herzog_2019', False)

### Dataset metadata and labels

- `d.metadata` — dict with dataset/publication/collection info
- `d.labels` — `openalex_id` → `{doi, pmid, lens_id, label_included, ...}`
- `d.cite` — citation string for this dataset
- `d.summary()` — quick statistics

In [6]:
pprint(d.metadata["data"])
pprint(d.metadata["publication"]["title"])

{'doi': '10.5281/zenodo.3625931', 'n_records': 2897, 'n_records_included': 26}
('Comparative effectiveness of common therapies for Wilson disease: A '
 'systematic review and meta‐analysis of controlled studies')


Eligibility criteria (inclusion/exclusion) for the review are stored under `d.metadata["publication"]["eligibility_criteria"]` — free text from the original systematic review, not split into separate inclusion/exclusion fields.

In [7]:
print(d.metadata["publication"]["eligibility_criteria"])

We included WD patients of any age or stage. The study drug had to be one of four established therapies, namely DPen, trientine, TTM or Zn. The control could be placebo, no treatment or any other treatment that does not include the respective study drug (eg Zn vs trientine was allowed, Zn 50 mg vs Zn 100 mg was not allowed). Concomitant therapies had to be identical in the compared treatment arms (eg trientine plus Zn vs TTM plus Zn). Comparisons between monotherapy and combination therapy regimens that included the respective monotherapy drug (eg DPen plus Zn vs Zn) have been analysed elsewhere and were not considered any further here. We included studies that reported all‐cause mortality, orthotopic liver transplantation (OLT), neurological symptoms (eg dystonia, dysarthria, cognitive decline, drooling, tremor, gait disturbance, chorea, seizure, psychosis), liver‐related symptoms (eg icterus, ascites, steatosis, fibrosis, mild hepatitis, acute liver failure, cirrhosis, serum transami

In [8]:
# First 3 label entries
dict(list(d.labels.items())[:3])

{'https://openalex.org/w2093266833': {'doi': 'https://doi.org/10.1016/s0022-2143(03)00027-1',
  'pmid': 'https://pubmed.ncbi.nlm.nih.gov/12819634',
  'lens_id': None,
  'label_included': 0,
  'label_abstract_included': 0},
 'https://openalex.org/w2409884690': {'doi': None,
  'pmid': 'https://pubmed.ncbi.nlm.nih.gov/11234295',
  'lens_id': None,
  'label_included': 0,
  'label_abstract_included': 0},
 'https://openalex.org/w2799651202': {'doi': 'https://doi.org/10.1016/s0168-8278(18)30431-8',
  'pmid': None,
  'lens_id': None,
  'label_included': 0,
  'label_abstract_included': 0}}

In [9]:
print(d.cite)

Appenzeller‐Herzog, C., Mathes, T., Heeres, M. L. S., Weiss, K. H., Houwen, R. H. J., & Ewald, H. (2019). Comparative effectiveness of common therapies for Wilson disease: A systematic review and meta‐analysis of controlled studies. Liver International, 39(11), 2136–2152. Portico. https://doi.org/10.1111/liv.14179



In [10]:
pprint(d.summary())

{'inclusion_rate': 0.008977900552486187,
 'languages': Counter({'en': 565,
                       'es': 7,
                       'pt': 5,
                       'fr': 2,
                       'it': 1,
                       'lv': 1,
                       'et': 1}),
 'n_excluded': 2870,
 'n_included': 26,
 'n_total': 2896,
 'name': 'Appenzeller-Herzog_2019',
 'primary_topics': ['Trace Elements in Health',
                    'Liver Disease and Transplantation',
                    'Congenital gastrointestinal and neural anomalies',
                    'Dermatological and Skeletal Disorders',
                    'Liver Disease Diagnosis and Treatment'],
 'year_range': (1964, 2019)}


## Export to DataFrame

For SYNERGY+, only open-access works with a valid abstract (≥ 20 words or ≥ 100 characters) are included.

In [11]:
df = d.to_frame()  # title + abstract
df.head()

,openalex_id,doi,pmid,lens_id,title,abstract,label_included,label_abstract_included
0,https://openalex.org/w1999503276,https://doi.org/10.1053/j.gastro.2006.04.048,https://pubmed.ncbi.nlm.nih.gov/16831610,NaN,American Gastroenterological Association Insti...,This document presents the official recommenda...,0,0
1,https://openalex.org/w2587973761,https://doi.org/10.1186/s12883-017-0818-1,https://pubmed.ncbi.nlm.nih.gov/28212618,NaN,Clinical features and outcome in patients with...,Background: Wilson 's disease with osseomuscul...,0,0
2,https://openalex.org/w2072870542,https://doi.org/10.1007/s10534-013-9694-3,https://pubmed.ncbi.nlm.nih.gov/24368744,NaN,Treatment with d-penicillamine or zinc sulphat...,Copper accumulation in tissues due to a bialle...,0,1
3,https://openalex.org/w2585486787,https://doi.org/10.1080/23262133.2016.1271495,NaN,NaN,Zeb2: Inhibiting the inhibitors in Schwann cells,Development of Schwann cells is tightly regula...,0,0
4,https://openalex.org/w4292495816,https://doi.org/10.1007/s00415-014-7337-4,NaN,NaN,Joint Congress of European Neurology 31 May–3 ...,Introduction: Wilson's disease (WD) is caused ...,0,0


In [12]:
df_extended = d.to_frame(vars="extended")  # all OpenAlex fields
print(df_extended.shape)
df_extended.head()

(582, 35)


,openalex_id,doi,pmid,lens_id,title,abstract,abstract_original,publication_year,publication_date,type,...,topics,keywords,mesh,sustainable_development_goals,indexed_in,referenced_works,related_works,counts_by_year,label_included,label_abstract_included
0,https://openalex.org/w1999503276,https://doi.org/10.1053/j.gastro.2006.04.048,https://pubmed.ncbi.nlm.nih.gov/16831610,NaN,American Gastroenterological Association Insti...,This document presents the official recommenda...,NaN,2006,2006-07-01,review,...,"[{'id': 'https://openalex.org/T11964', 'name':...","[Position statement, Medicine, Pregnancy, Stat...",[{'descriptor_name': 'Academies and Institutes...,"[{'id': 'https://metadata.un.org/sdg/3', 'name...","[crossref, pubmed]","[https://openalex.org/W2129773053, https://ope...","[https://openalex.org/W2475705533, https://ope...","[{'year': 2024, 'cited_by_count': 10}, {'year'...",0,0
1,https://openalex.org/w2587973761,https://doi.org/10.1186/s12883-017-0818-1,https://pubmed.ncbi.nlm.nih.gov/28212618,NaN,Clinical features and outcome in patients with...,Background: Wilson 's disease with osseomuscul...,NaN,2017,2017-02-17,article,...,"[{'id': 'https://openalex.org/T10915', 'name':...","[Medicine, Neurology, Neurosurgery, Neurochemi...","[{'descriptor_name': 'Abdomen', 'qualifier_nam...","[{'id': 'https://metadata.un.org/sdg/3', 'name...","[crossref, doaj, pubmed]","[https://openalex.org/W4250258618, https://ope...","[https://openalex.org/W341849362, https://open...","[{'year': 2025, 'cited_by_count': 1}, {'year':...",0,0
2,https://openalex.org/w2072870542,https://doi.org/10.1007/s10534-013-9694-3,https://pubmed.ncbi.nlm.nih.gov/24368744,NaN,Treatment with d-penicillamine or zinc sulphat...,Copper accumulation in tissues due to a bialle...,Copper accumulation in tissues due to a bialle...,2013,2013-12-24,article,...,"[{'id': 'https://openalex.org/T10915', 'name':...","[Glutathione, Antioxidant, Glutathione peroxid...","[{'descriptor_name': 'Adult', 'qualifier_name'...","[{'id': 'https://metadata.un.org/sdg/3', 'name...","[crossref, pubmed]","[https://openalex.org/W2143190548, https://ope...","[https://openalex.org/W2074324989, https://ope...","[{'year': 2024, 'cited_by_count': 1}, {'year':...",0,1
3,https://openalex.org/w2585486787,https://doi.org/10.1080/23262133.2016.1271495,NaN,NaN,Zeb2: Inhibiting the inhibitors in Schwann cells,Development of Schwann cells is tightly regula...,Development of Schwann cells is tightly regula...,2017,2017-01-01,article,...,"[{'id': 'https://openalex.org/T12331', 'name':...","[Biology, Schwann cell, Cell biology, Transcri...",[],"[{'id': 'https://metadata.un.org/sdg/3', 'name...","[crossref, pubmed]","[https://openalex.org/W1960355435, https://ope...","[https://openalex.org/W2017371576, https://ope...","[{'year': 2025, 'cited_by_count': 2}, {'year':...",0,0
4,https://openalex.org/w4292495816,https://doi.org/10.1007/s00415-014-7337-4,NaN,NaN,Joint Congress of European Neurology 31 May–3 ...,Introduction: Wilson's disease (WD) is caused ...,NaN,2014,2014-05-01,article,...,"[{'id': 'https://openalex.org/T12854', 'name':...","[Neurology, Neuroradiology, Joint (building), ...","[{'descriptor_name': 'Animals', 'qualifier_nam...",[],"[crossref, pubmed]",[],"[https://openalex.org/W2120185406, https://ope...","[{'year': 2023, 'cited_by_count': 1}, {'year':...",0,0


In [13]:
df_custom = d.to_frame(vars=["cited_by_count", "publication_year"])
df_custom.head()

,openalex_id,doi,pmid,lens_id,cited_by_count,publication_year,label_included,label_abstract_included
0,https://openalex.org/w1999503276,https://doi.org/10.1053/j.gastro.2006.04.048,https://pubmed.ncbi.nlm.nih.gov/16831610,NaN,139,2006,0,0
1,https://openalex.org/w2587973761,https://doi.org/10.1186/s12883-017-0818-1,https://pubmed.ncbi.nlm.nih.gov/28212618,NaN,27,2017,0,0
2,https://openalex.org/w2072870542,https://doi.org/10.1007/s10534-013-9694-3,https://pubmed.ncbi.nlm.nih.gov/24368744,NaN,19,2013,0,1
3,https://openalex.org/w2585486787,https://doi.org/10.1080/23262133.2016.1271495,NaN,NaN,8,2017,0,0
4,https://openalex.org/w4292495816,https://doi.org/10.1007/s00415-014-7337-4,NaN,NaN,4,2014,0,0


## Export to dict

In [14]:
records = d.to_dict()  # openalex_id -> record dict
pprint(list(records.values())[0])

{'abstract': 'This document presents the official recommendations of the '
             'American Gastroenterological Association (AGA) Institute on "Use '
             'of Gastrointestinal Medications in Pregnancy." It was approved '
             'by the Clinical Practice and Economics Committee on February 22, '
             '2006, and by the AGA Institute Governing Board on April 20, '
             '2006.',
 'doi': 'https://doi.org/10.1053/j.gastro.2006.04.048',
 'label_abstract_included': 0,
 'label_included': 0,
 'lens_id': None,
 'pmid': 'https://pubmed.ncbi.nlm.nih.gov/16831610',
 'title': 'American Gastroenterological Association Institute Medical Position '
          'Statement on the Use of Gastrointestinal Medications in Pregnancy'}


In [15]:
records_custom = d.to_dict(vars=["title", "referenced_works"])
pprint(list(records_custom.values())[0])

{'doi': 'https://doi.org/10.1053/j.gastro.2006.04.048',
 'label_abstract_included': 0,
 'label_included': 0,
 'lens_id': None,
 'pmid': 'https://pubmed.ncbi.nlm.nih.gov/16831610',
 'referenced_works': ['https://openalex.org/W2129773053',
                      'https://openalex.org/W1966227297',
                      'https://openalex.org/W6600611573',
                      'https://openalex.org/W2020723789',
                      'https://openalex.org/W6675288774',
                      'https://openalex.org/W2106233388',
                      'https://openalex.org/W2169166270',
                      'https://openalex.org/W2070542961',
                      'https://openalex.org/W2103134407',
                      'https://openalex.org/W15283199',
                      'https://openalex.org/W1679356500'],
 'title': 'American Gastroenterological Association Institute Medical Position '
          'Statement on the Use of Gastrointestinal Medications in Pregnancy'}


## Iterate over works

`d.iter()` yields `(work, label_included)` pairs. By default each work is validated against the OpenAlex Pydantic model (`WorkModel`); pass `validate=False` to skip validation for speed, in which case `work` is a plain `pyalex.Work` (dict-like).

In [16]:
for work, label in d.iter():
    print(type(work).__name__, "-", work.title[:60], "- label:", label)
    break

WorkModel - American Gastroenterological Association Institute Medical P - label: 0


In [17]:
for work, label in d.iter(validate=False):
    print(type(work).__name__, "-", work["title"][:60], "- label:", label)
    break

Work - American Gastroenterological Association Institute Medical P - label: 0


## Available `--vars` / extractor fields

These are the field names accepted by `to_frame()`, `to_dict()`, and `-v/--vars` on the CLI.

In [18]:
sorted(WORK_EXTRACTORS)

['abstract',
 'abstract_original',
 'author_names',
 'authorships',
 'cited_by_count',
 'counts_by_year',
 'fwci',
 'indexed_in',
 'is_oa',
 'is_paratext',
 'is_retracted',
 'journal_name',
 'keywords',
 'language',
 'language_fasttext',
 'mesh',
 'oa_status',
 'primary_topic_domain',
 'primary_topic_field',
 'primary_topic_name',
 'publication_date',
 'publication_year',
 'referenced_works',
 'referenced_works_count',
 'related_works',
 'sustainable_development_goals',
 'title',
 'topics',
 'type']

## Build one big dataset

Combine the default (title + abstract) frame of every dataset in the train split into a single DataFrame.

In [19]:
train_frame = pd.concat(
    [Dataset(name).to_frame() for name in train_names[:5]],
    axis=0,
    keys=train_names[:5],
    names=["dataset", None],
)
train_frame.head()

openalex_id  \
dataset                                          
Adamo_2021 0  https://openalex.org/w2259905321   
           1  https://openalex.org/w2156690657   
           2  https://openalex.org/w2617456871   
           3   https://openalex.org/w290581399   
           4  https://openalex.org/w2107495959   

                                                       doi  pmid  \
dataset                                                            
Adamo_2021 0           https://doi.org/10.5277/e-inf150104  None   
           1  https://doi.org/10.1007/978-3-540-69534-9_38  None   
           2  https://doi.org/10.1007/978-3-319-59536-8_28  None   
           3  https://doi.org/10.1007/978-3-319-19069-3_30  None   
           4            https://doi.org/10.1007/bf03192370  None   

                          lens_id  \
dataset                             
Adamo_2021 0                  NaN   
           1  049-919-799-175-639   
           2  084-050-799-604-086   
           3  064-229-162-110-606   
           4                  NaN   

                                                          title  \
dataset                                                           
Adamo_2021 0  An Approach to Assessing the Quality of Busine...   
           1  On Modeling and Analyzing Cost Factors in Info...   
           2  Predictive Business Process Monitoring Conside...   
           3  Modelling Service Level Agreements for Busines...   
           4  Analyzing requirements of knowledge management...   

                                                       abstract  \
dataset                                                           
Adamo_2021 0  Introduction: The quality of business process ...   
           1  Introducing enterprise information systems(EIS...   
           2  Predictive business process monitoring aims at...   
           3  Many proposals to model service level agreemen...   
           4  Knowledge Management (KM) is considered by man...   

              label_included  label_abstract_included  
dataset                                                
Adamo_2021 0               0                      NaN  
           1               0                      NaN  
           2               0                      NaN  
           3               0                      NaN  
           4               0                      NaN